In [1]:
import pandas as pd

df = pd.read_parquet("../data/labeled/labeled_500.parquet")

print("Shape:", df.shape)
print("\n--- COLUMNS ---")
print(df.columns.tolist())
print("\n--- FIRST 3 ROWS ---")
print(df.head(3).to_string())
print("\n--- DTYPES ---")
print(df.dtypes)

Shape: (500, 6)

--- COLUMNS ---
['order_id', 'review_score', 'review_clean', 'sentiment', 'theme', 'journey_stage']

--- FIRST 3 ROWS ---
                           order_id  review_score                                                                                     review_clean sentiment     theme  journey_stage
0  ec082d3ffd103096f9274b6247217ff9             1    I bought 2 items...a hammock and table with 2 children's chairs. I only received the hammock.  positive  delivery   pre-purchase
1  ec082d3ffd103096f9274b6247217ff9             1    I bought 2 items...a hammock and table with 2 children's chairs. I only received the hammock.  negative  delivery  post-purchase
2  610cb5d8ac9938a7e80ee9e3b9c04be5             1  counterfeit product, and one of mine listens, but out of the two I bought, it doesn't work.....  negative   quality  post-purchase

--- DTYPES ---
order_id         object
review_score      int64
review_clean     object
sentiment        object
theme            obje

In [2]:
Shape: (500, 6)

--- COLUMNS ---
['order_id', 'review_score', 'review_clean', 'sentiment', 'theme', 'journey_stage']

--- FIRST 3 ROWS ---
                           order_id  review_score                                                                                     review_clean sentiment     theme  journey_stage
0  ec082d3ffd103096f9274b6247217ff9             1    I bought 2 items...a hammock and table with 2 children's chairs. I only received the hammock.  positive  delivery   pre-purchase
1  ec082d3ffd103096f9274b6247217ff9             1    I bought 2 items...a hammock and table with 2 children's chairs. I only received the hammock.  negative  delivery  post-purchase
2  610cb5d8ac9938a7e80ee9e3b9c04be5             1  counterfeit product, and one of mine listens, but out of the two I bought, it doesn't work.....  negative   quality  post-purchase

--- DTYPES ---
order_id         object
review_score      int64
review_clean     object
sentiment        object
theme            object
journey_stage    object
dtype: object

SyntaxError: unterminated string literal (detected at line 8) (1158450587.py, line 8)

In [3]:
print("Total rows:", len(df))
print("Distinct order_ids:", df['order_id'].nunique())
print("Distinct review texts:", df['review_clean'].nunique())

dupes = df[df.duplicated('order_id', keep=False)].sort_values('order_id')
print("\nDuplicated rows:", len(dupes))
print(dupes[['order_id','review_score','sentiment','theme','journey_stage']].head(20).to_string())

print("\n--- SENTIMENT ---"); print(df['sentiment'].value_counts())
print("\n--- THEME ---"); print(df['theme'].value_counts())
print("\n--- JOURNEY ---"); print(df['journey_stage'].value_counts())

print("\nPositive sentiment on 1-2 star reviews:")
print(df[(df['sentiment']=='positive') & (df['review_score']<=2)].shape[0])

Total rows: 500
Distinct order_ids: 499
Distinct review texts: 498

Duplicated rows: 2
                           order_id  review_score sentiment     theme  journey_stage
0  ec082d3ffd103096f9274b6247217ff9             1  positive  delivery   pre-purchase
1  ec082d3ffd103096f9274b6247217ff9             1  negative  delivery  post-purchase

--- SENTIMENT ---
sentiment
negative    271
positive    176
neutral      53
Name: count, dtype: int64

--- THEME ---
theme
delivery    236
quality     159
other        51
service      37
returns      12
price         5
Name: count, dtype: int64

--- JOURNEY ---
journey_stage
post-purchase    283
delivery         208
pre-purchase       9
Name: count, dtype: int64

Positive sentiment on 1-2 star reviews:
5


In [4]:
# the 5 suspect positives
print("--- POSITIVE on 1-2 star (check for mislabels) ---")
print(df[(df['sentiment']=='positive') & (df['review_score']<=2)][['review_score','review_clean','sentiment','theme']].to_string())

# the duplicate
print("\n--- DUPLICATE ---")
print(df[df['order_id']=='ec082d3ffd103096f9274b6247217ff9'][['review_clean','sentiment','theme','journey_stage']].to_string())

--- POSITIVE on 1-2 star (check for mislabels) ---
     review_score                                                                                                                                review_clean sentiment     theme
0               1                                               I bought 2 items...a hammock and table with 2 children's chairs. I only received the hammock.  positive  delivery
39              1                 I believe I didn't receive it due to the strike. But I do not recommend Lannister due to the sale of defective televisions.  positive  delivery
54              1       It took almost 15 days to say that they didn't have the product in stock... a huge lack of responsibility on the part of the company.  positive  delivery
86              1  The order arrived late, the website didn't show details of the object, it wasn't what I was looking for. It was a waste of time and money.  positive  delivery
100             1                   I didn't receive everyt

In [5]:
import pandas as pd
print(pd.crosstab(df['review_score'], df['sentiment']))

sentiment     negative  neutral  positive
review_score                             
1                   90        6         5
2                   90       10         0
3                   63       18        19
4                   27       14        58
5                    1        5        94


In [6]:
# drop the duplicate (keep the correct negative row)
df = df[~((df['order_id']=='ec082d3ffd103096f9274b6247217ff9') & (df['sentiment']=='positive'))].copy()

# relabel the 4 remaining clear errors
df.loc[[39, 54, 86, 100], 'sentiment'] = 'negative'

print("Rows now:", len(df))   # expect 499
print(df['sentiment'].value_counts())

Rows now: 499
sentiment
negative    275
positive    171
neutral      53
Name: count, dtype: int64


## Cell A: collapse the dead theme class + save the cleaned labels ##

In [7]:
# price has 5 samples, too few to learn. Fold it into 'other'.
df['theme'] = df['theme'].replace('price', 'other')

# save the cleaned labels so this work survives a kernel restart
df.to_parquet("../data/labeled/labeled_clean.parquet", index=False)

print("Theme counts after collapse:")
print(df['theme'].value_counts())

Theme counts after collapse:
theme
delivery    235
quality     159
other        56
service      37
returns      12
Name: count, dtype: int64


## Cell B: train/test split ##

In [8]:
from sklearn.model_selection import train_test_split

X = df['review_clean']

# separate splits because sentiment and theme have different class balances
Xs_train, Xs_test, ys_train, ys_test = train_test_split(
    X, df['sentiment'], test_size=0.2, stratify=df['sentiment'], random_state=42)

Xt_train, Xt_test, yt_train, yt_test = train_test_split(
    X, df['theme'], test_size=0.2, stratify=df['theme'], random_state=42)

print("Sentiment train/test:", len(Xs_train), len(Xs_test))
print("Theme train/test:", len(Xt_train), len(Xt_test))

Sentiment train/test: 399 100
Theme train/test: 399 100


## Cell C: sentiment baseline ##

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

sentiment_model = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))
])

sentiment_model.fit(Xs_train, ys_train)
preds = sentiment_model.predict(Xs_test)
print(classification_report(ys_test, preds))

              precision    recall  f1-score   support

    negative       0.84      0.89      0.87        55
     neutral       0.50      0.36      0.42        11
    positive       0.91      0.91      0.91        34

    accuracy                           0.84       100
   macro avg       0.75      0.72      0.73       100
weighted avg       0.83      0.84      0.83       100



In [10]:
theme_model = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))
])

theme_model.fit(Xt_train, yt_train)
theme_preds = theme_model.predict(Xt_test)
print(classification_report(yt_test, theme_preds))

              precision    recall  f1-score   support

    delivery       0.70      0.83      0.76        47
       other       0.50      0.36      0.42        11
     quality       0.63      0.59      0.61        32
     returns       0.50      0.50      0.50         2
     service       0.50      0.25      0.33         8

    accuracy                           0.65       100
   macro avg       0.57      0.51      0.52       100
weighted avg       0.63      0.65      0.64       100



## Cell D: refit on full labels + apply to all reviews ##

In [11]:
import joblib

# refit both on ALL labeled data (you've already measured them)
sentiment_model.fit(X, df['sentiment'])
theme_model.fit(X, df['theme'])

# load all translated reviews
full = pd.read_parquet("../data/processed/olist_reviews_translated.parquet")
print("Columns:", full.columns.tolist())   # confirm 'review_clean' exists
print("Rows:", len(full))

# predict
full['sentiment_pred'] = sentiment_model.predict(full['review_clean'])
full['theme_pred'] = theme_model.predict(full['review_clean'])

# sanity check: predicted sentiment should track star rating
print(pd.crosstab(full['review_score'], full['sentiment_pred']))

Columns: ['order_id', 'review_pt', 'review_en', 'review_score', 'review_clean', 'review_preprocessed']
Rows: 11997
sentiment_pred  negative  neutral  positive
review_score                               
1                   7702      426       187
2                   1779      132        94
3                    116       33        30
4                    132       44       183
5                    124       32       983


## Cell E: save models + scored data ##

In [12]:
joblib.dump(sentiment_model, "../models/sentiment_classifier.pkl")
joblib.dump(theme_model, "../models/theme_classifier.pkl")
full.to_parquet("../data/processed/olist_reviews_scored.parquet", index=False)
print("Saved models and scored reviews.")

Saved models and scored reviews.
